In [ ]:
from IPython.display import clear_output
import warnings
warnings.filterwarnings('ignore', message='.*invalid escape sequence.*', category=SyntaxWarning)
warnings.filterwarnings('ignore', category=RuntimeWarning)
from pathlib import Path
import numpy as np
import tifffile
import skimage
import colormaps as cmaps
import scipy
import skan
from matplotlib_scalebar.scalebar import ScaleBar
from tqdm import tqdm
import scienceplots
import matplotlib.pyplot as plt
from matplotlib import rcParams
plt.style.use('nature')
rcParams['font.family'] = 'Arial'

from joblib import Parallel, delayed
import multiprocessing as mp

In [ ]:
def plot_frame(frame, cblabel, title=None, cmap=cmaps.gray, vmin=0.0, vmax=1.0, cbmin=0, cbmax=1, scalebar_size_um=50.0, voxel_resolution=(1,1,1), show_title=True):
    if vmin is None and vmax is None: plt.imshow(frame, cmap=cmap, extent=(0, frame.shape[1]*voxel_resolution[2], 0, frame.shape[0]*voxel_resolution[1]))
    else: plt.imshow(frame, cmap=cmap, vmin=vmin, vmax=vmax, extent=(0, frame.shape[1]*voxel_resolution[2], 0, frame.shape[0]*voxel_resolution[1]))
    plt.axis(False)
    cb = plt.colorbar(fraction=0.046, pad=0.04)
    if np.sign(cbmax) == 1: 
        cb.set_ticks([cbmin, cbmax], minor=False)
        cb.set_ticklabels([cbmin,cbmax], fontsize=6, minor=False)
        cb.set_ticks([(cbmax-cbmin)/2], minor=True)
    else: 
        cb.set_ticks([cbmin, -cbmax], minor=False)
        cb.set_ticklabels([cbmax, cbmin], fontsize=6, minor=False)
        cb.set_ticks([(-cbmax+cbmin)/2], minor=True)
    cb.set_ticklabels([f'\n{cblabel}'], fontsize=6, minor=True, rotation='vertical', va='center')
    cb.ax.yaxis.set_tick_params('major', width=0.5)
    cb.ax.yaxis.set_tick_params('minor', width=0.0, length=0.0)
    cb.outline.set_visible(True)
    cb.outline.set_linewidth(0.5)
    scalebar = ScaleBar(scalebar_size_um, units='um', length_fraction=scalebar_size_um*voxel_resolution[2]/frame.shape[1], location='lower right', pad=0.05, border_pad=0.1, box_alpha=0, scale_loc='none', label_loc='none', width_fraction=0.015)
    plt.gca().add_artist(scalebar)
    if title is not None and show_title: plt.title(title)
    
def extract_depth_frame(mask_3d, z_res=2.0):
    mask_depth_3d = mask_3d * np.arange(mask_3d.shape[0], 0, -1, dtype=float)[:, None, None]
    mask_depth_3d_tmp = mask_depth_3d.copy()
    mask_depth_3d_tmp[~mask_3d] = 0
    mask_depth_3d[~mask_3d] = np.nan
    return np.nanmax(mask_depth_3d, axis=0)*z_res

def extract_colorized_frame(mask_3d, stack_3d):
    # 3D skeleton (use the 3D skeletonizer)
    skel = skimage.morphology.skeletonize(mask_3d.astype(np.uint8)).astype(bool)

    # top-most skeleton z index per (y,x)
    skel_depth = skel * np.arange(skel.shape[0], 0, -1, dtype=float)[:, None, None]
    skel_depth_tmp = skel_depth.copy()
    skel_depth_tmp[~skel] = 0
    max_inds = np.nanargmax(skel_depth_tmp, axis=0)
    rows, cols = np.indices(max_inds.shape)

    # isolate skeleton intensities (use float so we can assign NaN)
    stack_3d_tmp = stack_3d.astype(float).copy()
    stack_3d_tmp[~skel] = np.nan
    stack_3d_tmp[~mask_3d] = np.nan

    stack_2d = stack_3d_tmp[max_inds, rows, cols]

    # ensure pixels with no skeleton remain NaN
    skel_any = np.any(skel, axis=0)
    stack_2d[~skel_any] = np.nan
    return stack_2d

def _rtn(val, nearest=25):
    return int(round(val / nearest) * nearest)

def view_processing(sample_name,
                    raw_stack, denoised_stack, segmented_mask, 
                    distance_transform_stack, radius_filtered_mask, filtered_mask,
                    voxel_resolution=(1,1,1),
                    show_titles=True, show_sample_name=True, scalebar_size_um=50, 
                    save_fig=False):
    
    
    plt.figure(figsize=(3.37*2, 3.37*1.25), dpi=600)
    # Raw Stack - Maximum Z-Projection
    plt.subplot(2, 3, 1)
    plot_frame(
        frame=raw_stack.max(axis=0),
        cblabel='Normalized Intensity',
        title = 'Raw',
        cmap=cmaps.neutral_r,
        vmin=0.0, vmax=1.0,
        cbmin=0.0, cbmax=1.0,
        scalebar_size_um=scalebar_size_um, voxel_resolution=voxel_resolution,
        show_title = show_titles
    )
    # Denoised Stack - Maximum Z-Projection
    plt.subplot(2, 3, 2)
    plot_frame(
        frame=denoised_stack.max(axis=0),
        cblabel='Normalized Intensity',
        title = 'Denoised',
        cmap=cmaps.neutral_r,
        vmin=0.0, vmax=1.0,
        cbmin=0.0, cbmax=1.0,
        scalebar_size_um=scalebar_size_um, voxel_resolution=voxel_resolution,
        show_title = show_titles
    )
    # Original Segmented Mask - Maximum Z-Projection; colored by depth
    plt.subplot(2, 3, 3)
    
    plot_frame(
        frame=extract_depth_frame(segmented_mask, z_res=voxel_resolution[0]),
        cblabel='Relative Depth (µm)',
        title = 'Original\nSegmentation Mask',
        cmap=cmaps.wildfire,
        vmin=0.0, vmax=_rtn(segmented_mask.shape[0]*voxel_resolution[0], nearest=25),
        cbmin=0.0, cbmax=-_rtn(segmented_mask.shape[0]*voxel_resolution[0], nearest=25),
        scalebar_size_um=scalebar_size_um, voxel_resolution=voxel_resolution,
        show_title = show_titles
    )
    # Distance-Transformed Segmentation Mask - Maximum Z-Projection; colored by radius
    # plt.subplot(2, 3, 4)
    # dtsm = extract_colorized_frame(segmented_mask, distance_transform_stack)
    
    # plot_frame(
    #     frame=dtsm,
    #     cblabel='Local Radius (µm)',
    #     title = 'Skeletonized EDT of\nSegmentation Mask',
    #     cmap=cmaps.cubehelix3_16.cut(1/16, 'right'),
    #     vmin=0.0, vmax=16,
    #     cbmin=0.0, cbmax=16,
    #     scalebar_size_um=scalebar_size_um, voxel_resolution=voxel_resolution,
    #     show_title = show_titles
    # )
    # Distance-Transformed Segmentation Mask - Maximum Z-Projection; colored by radius
    plt.subplot(2, 3, 6)
    plot_frame(
        frame=extract_depth_frame(radius_filtered_mask, z_res=voxel_resolution[0]),
        cblabel='Relative Depth (µm)',
        title = 'Radius Filtered\nSegmentation Mask',
        cmap=cmaps.wildfire,
        vmin=0.0, vmax=_rtn(radius_filtered_mask.shape[0]*voxel_resolution[0], nearest=25),
        cbmin=0.0, cbmax=-_rtn(radius_filtered_mask.shape[0]*voxel_resolution[0], nearest=25),
        scalebar_size_um=scalebar_size_um, voxel_resolution=voxel_resolution,
        show_title = show_titles
    )
    # Distance-Transformed Segmentation Mask - Maximum Z-Projection; colored by radius
    plt.subplot(2, 3, 5)
    plot_frame(
        frame=extract_depth_frame(filtered_mask, z_res=voxel_resolution[0]),
        cblabel='Relative Depth (µm)',
        title = 'Small-Object Filtered\nSegmentation Mask',
        cmap=cmaps.wildfire,
        vmin=0.0, vmax=_rtn(filtered_mask.shape[0]*voxel_resolution[0], nearest=25),
        cbmin=0.0, cbmax=-_rtn(filtered_mask.shape[0]*voxel_resolution[0], nearest=25),
        scalebar_size_um=scalebar_size_um, voxel_resolution=voxel_resolution,
        show_title = show_titles
    )
    # Distance-Transformed Segmentation Mask - Maximum Z-Projection; colored by radius
    plt.subplot(2, 3, 4)
    plt.imshow(skimage.morphology.skeletonize(filtered_mask).max(axis=0), cmap='binary')
    plt.title('Skeletonized\nFiltered Mask')
    plt.axis(False)
    
    if show_sample_name: plt.suptitle(sample_name.split('_SPRT')[0].replace('_', ' '), weight='bold')
    plt.tight_layout(w_pad=5.0, h_pad=3.0, pad=2.0)
    if save_fig: plt.savefig('output.png', dpi=600)
    plt.show()

In [ ]:
def radius_filtration(segmented_mask: bool, voxel_resolution: tuple = (1, 1, 1),
                      radial_threshold_um: float = 0.0, length_threshold_um: float = 0.0):
    r_thr = radial_threshold_um
    l_thr = length_threshold_um
    structure = np.ones((3, 3, 3), dtype=bool)
    segmented_mask = scipy.ndimage.binary_fill_holes(segmented_mask, structure=skimage.morphology.ball(1))

    local_radius_um = scipy.ndimage.distance_transform_edt(segmented_mask, sampling=voxel_resolution)
    thick_core = local_radius_um >= r_thr
    core_labels, n_cores = scipy.ndimage.label(thick_core, structure=structure)

    candidate = np.zeros_like(segmented_mask, dtype=bool)
    if n_cores > 0:
        for reg in skimage.measure.regionprops(core_labels):
            if l_thr is not None:
                coords = reg.coords
                z0, z1 = coords[:, 0].min(), coords[:, 0].max() + 1
                y0, y1 = coords[:, 1].min(), coords[:, 1].max() + 1
                x0, x1 = coords[:, 2].min(), coords[:, 2].max() + 1
                reg_length = np.sqrt(((z1-z0)*voxel_resolution[0])**2 + ((y1-y0)*voxel_resolution[1])**2 + ((x1-x0)*voxel_resolution[2])**2)
                if reg_length >= float(l_thr): candidate[core_labels == reg.label] = True
            else: candidate[core_labels == reg.label] = True
    dist_to_candidate, inds = scipy.ndimage.distance_transform_edt(~candidate, sampling=voxel_resolution, return_indices=True)
    nearest_r = local_radius_um[inds[0], inds[1], inds[2]]
    removal_mask = dist_to_candidate <= (nearest_r + r_thr)
    filtered_mask = segmented_mask.copy()
    filtered_mask[removal_mask] = False
    return filtered_mask, local_radius_um

def remove_by_skeleton_length(mask3d: np.ndarray,
                              voxel_resolution_um=(1.0, 1.0, 1.0),
                              min_length_um: float = 30.0):
    structure = skimage.morphology.ball(1)
    labels, n = scipy.ndimage.label(mask3d, structure=structure)
    out = np.zeros_like(mask3d, dtype=bool)
    lengths = {}
    for lab in range(1, n + 1):
        comp = labels == lab
        if not np.any(comp):
            lengths[lab] = 0.0
            continue

        skel = skimage.morphology.skeletonize(comp.astype(np.uint8), method='lee').astype(bool)
        if not np.any(skel):
            lengths[lab] = 0.0
            continue

        if np.count_nonzero(skel) < 2:
            lengths[lab] = 0.0
            continue

        try:
            Ls = skan.csr.Skeleton(skel, spacing=voxel_resolution_um, source_image=comp).path_lengths()
            L = float(np.sum(Ls)) if len(Ls) else 0.0
        except ValueError:
            lengths[lab] = 0.0
            continue

    return out

def capillary_filtration(denoised_path: str, segmented_path: str, capillary_mask_path: str, 
                         voxel_resolution: tuple = (1, 1, 1), 
                         radius_threshold_um: float = 0.0, length_threshold_um: float = 0.0, min_volume_voxels: int = 0.0):
    
    denoised_stack = tifffile.TiffFile(denoised_path).asarray()
    denoised_stack = skimage.exposure.rescale_intensity(denoised_stack, out_range=(0, 1))
    segmented_mask = tifffile.TiffFile(segmented_path).asarray().astype(bool)

    radius_filtered_mask, distance_transform_stack = radius_filtration(segmented_mask, voxel_resolution, radius_threshold_um, length_threshold_um)
    filtered_mask = skimage.morphology.remove_small_objects(radius_filtered_mask, min_size=min_volume_voxels, connectivity=3)
    tifffile.imwrite(capillary_mask_path, filtered_mask.astype(np.uint8))
    return denoised_stack, segmented_mask, distance_transform_stack, radius_filtered_mask, filtered_mask

def collect_paths(dir_for_raw_stacks: str, dir_for_denoised_stacks: str, dir_for_segmented_masks: str, dir_for_filtered_masks: str):
    dir_raw = dir_for_raw_stacks
    dir_denoised = dir_for_denoised_stacks
    dir_segmented = dir_for_segmented_masks
    dir_filtered = dir_for_filtered_masks
    if dir_raw[-1] != '/': dir_raw += '/'
    if dir_denoised[-1] != '/': dir_denoised += '/'
    if dir_segmented[-1] != '/': dir_segmented += '/'
    if dir_filtered[-1] != '/': dir_filtered += '/'
    
    paths_raw = list(Path(dir_raw).glob('*.tif'))
    paths_raw = sorted([str(path) for path in paths_raw])
    paths_denoised = list(Path(dir_denoised).glob('*.tif'))
    paths_denoised = sorted([str(path) for path in paths_denoised])
    paths_segmented = [dir_segmented + str(Path(path).stem) + '_pred.tif' for path in paths_denoised]
    paths_filtered = [dir_filtered + str(Path(path).stem) + '_pred_clean.tif' for path in paths_denoised]
    paths = {
        'raw': paths_raw,
        'denoised': paths_denoised,
        'segmented': paths_segmented,
        'filtered': paths_filtered
    }
    print('SAMPLES')
    for i in range(len(paths['raw'])): print(f'{i:02d}:\t', str(Path(paths['raw'][i]).stem))
    for i in range(len(paths['denoised'])): print(f'{i:02d}:\t', str(Path(paths['denoised'][i]).stem))
    return paths

In [ ]:
def vessel_length_density(capillary_mask, voxel_resolution_um=(1,1,1), verbose=False, limit=None, return_lengths=False):
    Z, Y, X = capillary_mask.shape
    dz, dy, dx = voxel_resolution_um
    skeleton = skimage.morphology.skeletonize(capillary_mask, method='lee')
    lengths = skan.csr.Skeleton(skeleton, spacing=voxel_resolution_um, source_image=capillary_mask).path_lengths()
    if verbose: 
        print(np.median(lengths), np.mean(lengths), np.std(lengths))
    if return_lengths: return lengths
    else:
        if limit is None:
            return np.sum(lengths * 1E-6) / np.sum((Z*dz*1E-3) * (Y*dy*1E-3) * (X*dx*1E-3))
        else:
            return np.sum(lengths[lengths > limit] * 1E-6) / np.sum((Z*dz*1E-3) * (Y*dy*1E-3) * (X*dx*1E-3))

In [ ]:
dir_for_raw_stacks = '/Users/garfinkeljb/Documents/Finalized 5xFAD Paper Materials/all_raw'
dir_for_denoised_stacks = '/Users/garfinkeljb/Documents/Finalized 5xFAD Paper Materials/all_denoised'
dir_for_segmented_masks = '/Users/garfinkeljb/Documents/Finalized 5xFAD Paper Materials/all_segmented'
dir_for_filtered_masks = '/Users/garfinkeljb/Documents/Finalized 5xFAD Paper Materials/all_filtered'

voxel_resolution = (2.0, 1.0859, 1.0859) # (Z, Y, X) µm

radius_threshold_um = 4
length_threshold_um = 30
min_volume_voxels = int(np.pi*2**2)*12

In [ ]:
paths = collect_paths(dir_for_raw_stacks, dir_for_denoised_stacks, dir_for_segmented_masks, dir_for_filtered_masks)
print('\n\nFILTRATION PARAMETERS')
print(f'Minimum Object Volume:\t{min_volume_voxels} voxels')
print(f'Radius Threshold:\t{radius_threshold_um} µm')
print(f'Length Threshold:\t{length_threshold_um} µm')

In [ ]:
for i in range(len(paths['raw'])): print(f'{i:02d}:\t', str(Path(paths['raw'][i]).stem))

In [ ]:
results = Parallel(n_jobs=-1, backend='threading')(
    delayed(capillary_filtration)(
        paths['denoised'][i], paths['segmented'][i], paths['filtered'][i], 
        voxel_resolution,
        radius_threshold_um, length_threshold_um, min_volume_voxels, 
    ) for i in tqdm(range(len(paths['segmented'])))
)
raw_stacks = []
for path in paths['raw']: 
    raw_stack = tifffile.TiffFile(path).asarray()
    raw_stacks.append(skimage.exposure.rescale_intensity(raw_stack, out_range=(0, 1)))
    del raw_stack
denoised_stacks          = {i: r[0] for i, r in enumerate(results)}
segmented_masks          = {i: r[1] for i, r in enumerate(results)}
distance_transform_stack = {i: r[2] for i, r in enumerate(results)}
radius_filtered_masks    = {i: r[3] for i, r in enumerate(results)}
filtered_masks           = {i: r[4] for i, r in enumerate(results)}

In [ ]:
for sample_number in range(len(paths['denoised'])):
    sample_name = Path(paths['denoised'][sample_number]).stem
    view_processing(sample_name, 
                    raw_stacks[sample_number], denoised_stacks[sample_number], segmented_masks[sample_number], 
                    distance_transform_stack[sample_number], radius_filtered_masks[sample_number], filtered_masks[sample_number], 
                    voxel_resolution=voxel_resolution, show_titles=True, show_sample_name=True)

In [ ]:
vessel_densities = {
    '5xFAD': {'stems': [], 'values': []},
    'Control': {'stems': [], 'values': []}
}
for i in range(len(filtered_masks)):
    if 'TgNeg' in str(Path(paths['filtered'][i]).stem):
        vessel_densities['Control']['stems'].append(str(Path(paths['filtered'][i]).stem))
        vessel_densities['Control']['values'].append(vessel_length_density(filtered_masks[i], voxel_resolution_um=voxel_resolution))
    else: 
        vessel_densities['5xFAD']['stems'].append(str(Path(paths['filtered'][i]).stem))
        vessel_densities['5xFAD']['values'].append(vessel_length_density(filtered_masks[i], voxel_resolution_um=voxel_resolution))

print('5xFAD')
for i in range(len(vessel_densities['5xFAD']['stems'])): 
    print(vessel_densities['5xFAD']['stems'][i], '\t\t', vessel_densities['5xFAD']['values'][i])

print('\nControl')
for i in range(len(vessel_densities['Control']['stems'])): 
    print(vessel_densities['Control']['stems'][i], '\t\t', vessel_densities['Control']['values'][i])